In [44]:
# ========== 导入与 system prompt：定义「夏令营活动助手」的结构化 JSON 输出 ==========
# 练习目标：抓取 Springplat 落地页 → 用 Chat Completions 产出 bootcamp JSON → 再生成营销文案（流式）
# 和本课关系：Website scrape + system/user messages + response_format=json_object + stream=True

# 导入标准库 json：把模型返回的 JSON 字符串解析成 Python dict
import json
# 从 openai 导入 OpenAI 客户端类：调用云端 Chat Completions API
from openai import OpenAI
# 从同目录 scraper 导入抓取函数：用 HTTP/解析拿到网页正文（具体实现见 scraper.py）
from scraper import fetch_website_contents
# 从 IPython.display 导入展示工具：在笔记本里渲染 Markdown，并支持流式 update_display
from IPython.display import Markdown, display, update_display

# 创建 OpenAI 客户端；默认从环境变量 OPENAI_API_KEY 读密钥（需事先 load_dotenv 或导出）
openai = OpenAI()

# 云端小模型常量：后面 JSON 生成步骤用这个便宜、够用的模型
MODEL_GPT = "gpt-4o-mini"

# system prompt：规定角色、机构背景，并要求「必须按给定 JSON schema 回答」
# 注意：发给模型的英文指令保留原样——翻译会改变输出风格/字段行为
system_prompt = """
Your are an assistant to the principle tutor and Springplat Code Create.
Springplat Code Create is all about developing computational skills and character in young
students to enable them to not only thrive but to also make good life choices.
The summer bootcamps are fun, interactive and memorable events for the students.

You are to include Springplat Code Create contact detais and respond with JSON as follows:
{
    "bootcamps": [
        {
            "program": "Scratch",
            "location": "online",
            "theme": "Create an awesome platformer game!",
            "cost": "$35.00",
            "ages": "12-14 years",
            "days": "Every Saturdays",
            "dates": "May 23 - Jun 27",
            "time": "10:25am-11:25am EST",
        },
        {
            "program": "MIT App Inventor",
            "location": "online",
            "theme": "A game card collectors app!",
            "cost": "$35.00",
            "ages": "12-14 years",
            "days": "Every Saturdays",
            "dates": "May 23 - Jun 27",
            "time": "11:30am-12:30am EST"
        }
    ],
    "contacts":[]
}
"""


In [45]:
# ========== 抓取落地页 + 调用 API 生成夏令营活动 JSON ==========

# 抓取 Springplat Code Create 官网正文；URL 保持原样（可运行 / 影响抓取目标）
web_content = fetch_website_contents("https://create.springplat.com/")
# 给网页正文加一个 Markdown 小标题，方便拼进 user prompt 当上下文
web_content_res = f"## SCC Landing Page:\n\n{web_content}\n"
# 打印抓到的内容，便于目视检查 scrape 是否成功、文本是否够用
print(web_content_res)

def create_bootcamp_event(webc):
    """根据网页上下文 + 活动需求，让模型返回结构化 bootcamp JSON。"""
    # user prompt：描述要创建的夏令营规格（年龄、时长、课程）；英文保留给模型
    user_prompt = """
    Create a summer bootcamp event with a fun theme for students age 7-10 for a 2 day
    Scratch coding weekend. A Fri evening (1 hr) followed by Sat morning (3 hrs).
    """
    # 把落地页正文追加到 user 消息后面，当作事实来源（联系方式、品牌语气等）
    user_prompt += webc
    # Chat Completions：system 定 schema/角色，user 定任务+网页；强制 json_object 输出
    response = openai.chat.completions.create(
        model=MODEL_GPT,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        # response_format：要求模型返回合法 JSON 对象（而不是随意 Markdown）
        response_format={"type": "json_object"}
    )
    # 取出第一条 choice 的文本内容（此时应是 JSON 字符串）
    result = response.choices[0].message.content
    # 解析 JSON → Python dict，供下一格营销文案使用
    content = json.loads(result)
    return content

# 执行：得到 bootcamps 结构化数据（含 program / theme / contacts 等字段）
bootcamps = create_bootcamp_event(web_content)


## SCC Landing Page:

Springplat Code Create

Kids Coding Program,
Build the Future with Purpose
A Student-Centered Coding Program Built on Character and Innovation.
We guide students (ages 7-17) to build their own apps and games within a supportive community that prioritizes character just as much as code
Register A Child
Is your child only consuming technology?
"I want my child to use tech productively, not just play games on devices for hours!"
But, i’m not even tech-savvy! How will i do this?
Kids also get frustrated coding alone — they need guidance, encouragement, and community.
Phew! See what i mean?
Register A Child
INTRODUCING...
Springplat Code Create
At Springplat Code Create, we believe that every child has the potential to build something meaningful with technology. Founded by a software engineer with 10+ years experience, our mission is to steward student-centered coding programs that:
Empower students to express their passions through app creation.
Equip them with comput

In [46]:
# ========== 基于活动 JSON：流式生成营销传单 + 社媒文案 ==========

# 步骤 1：重写 system / user 提示词（角色从「JSON 助手」切到「营销专家」）
# system：品牌定位一句话；发给模型的英文保留
system_prompt = "You are a marketing expert for Springplat Code Create. A Student-Centered Coding Program Built on Character and Innovation."
# user：要求产出传单 + 配套社媒帖；英文保留
user_prompt = """
    Create marketing fliers with accompany social media posts.
"""

# 把上一格的 bootcamps dict 序列化成 JSON 字符串，拼进 user 消息当素材
user_prompt += json.dumps(bootcamps)

# 组装 Chat Completions 所需的 messages 列表（system + user）
messages = [
    {"role":"system", "content": system_prompt},
    {"role":"user", "content": user_prompt}
] # fill this in

# 步骤 3：发起流式请求；模型名 gpt-4.1-mini 保持原样（与上一格 MODEL_GPT 不同，属刻意选择）
stream = openai.chat.completions.create(
    model="gpt-4.1-mini",
    messages=messages,
    stream=True
)
# 累积完整回复文本（边收边拼）
response = ""
# 先占位一个空 Markdown 显示句柄，后面用同一个 display_id 原地更新（打字机效果）
display_handle = display(Markdown(""), display_id=True)
# 遍历流式 chunk：取出 delta.content（可能为 None），拼进 response 并刷新显示
for chunk in stream:
    response += chunk.choices[0].delta.content or ''
    update_display(Markdown(response), display_id=display_handle.display_id)


Sure! Below are marketing flier content and social media posts for the Scratch Weekend Camp by Springplat Code Create.

---

### Marketing Flier Content

**Front Side:**

**Springplat Code Create**  
*Student-Centered Coding Program Built on Character and Innovation*

---

**Scratch Weekend Camp**  
**Adventure in Coding: Create Your Own Magical World!**

- Ages: 7-10 years  
- Location: Online (Learn from the comfort of home!)  
- Dates: July 14 - July 15  
- Days & Times:  
  - Friday: 5:00pm - 6:00pm EST  
  - Saturday: 10:00am - 1:00pm EST  
- Cost: $50.00  

---

What your child will learn:  
- Introduction to Scratch programming  
- Creative storytelling through coding  
- Build exciting interactive projects  
- Collaborate and innovate with peers  

---

**Register Now!**  
Limited spots available.

Contact us:  
📞 123-456-7890  
📧 info@springplatcodecreate.com  
🌐 www.springplatcodecreate.com

---

**Back Side:**

**Why Choose Springplat Code Create?**

- Student-centered learning focused on character & creativity  
- Experienced instructors passionate about coding and kids  
- Fun, engaging, and innovative curriculum  
- Safe and supportive online environment  

---

Unlock your child’s creativity this summer with coding!  
Join our Scratch Weekend Camp and watch their imagination come to life.

---

### Social Media Posts

**Facebook/Instagram Post:**

🌟 Calling all kids ages 7-10! 🌟  

Join our **Scratch Weekend Camp** and embark on an *Adventure in Coding* to create your very own magical world! 🧙‍♂️✨  

📅 July 14 - July 15  
🕔 Friday 5-6pm EST | Saturday 10am-1pm EST  
💻 Online | Only $50  

Spots are limited! Sign up today: www.springplatcodecreate.com  

#CodingForKids #ScratchCamp #SummerCamp #KidsWhoCode #STEM #SpringplatCodeCreate  

---

**Twitter Post:**

Kids 7-10! ✨ Join our Scratch Weekend Camp July 14-15. Create your own magical world through coding! Online, $50. Register now: www.springplatcodecreate.com #KidsCoding #ScratchCamp #STEMfun

---

Let me know if you want designs or specific file types for the fliers!